In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()

if not (NOTEBOOK_DIR / "_bootstrap.py").exists():
    candidates = [NOTEBOOK_DIR, NOTEBOOK_DIR.parent, NOTEBOOK_DIR.parent.parent]
    for candidate in candidates:
        if (candidate / "_bootstrap.py").exists():
            NOTEBOOK_DIR = candidate
            break

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from _bootstrap import PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, OUTPUT_DIR, RESULTS_DIR

print(PROJECT_ROOT)
print(RAW_DIR)
print(PROCESSED_DIR)
print(OUTPUT_DIR)
print(RESULTS_DIR)

/Users/rodrigue.lawson/VSCode Projects/lexcare-ai
/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/raw
/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/data/processed
/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/experiments/output
/Users/rodrigue.lawson/VSCode Projects/lexcare-ai/experiments/results


In [2]:
from app.repositories.source_registry import SourceRegistry
from app.services.hub_source_service import HubSourceService

registry = SourceRegistry(registry_path=str(PROJECT_ROOT / "data" / "source_registry.yaml"))
service = HubSourceService(source_registry=registry)

created = service.sync_sources()
print(created)

[]


In [3]:
hub_sources = service.repository.load_all()
hub_sources

[HubSource(source_key='31dfd5cb-98b9-438a-92ea-bbad81849ed4', source_id='gesetze-im-internet', source_name='Gesetze im Internet', created_at=datetime.datetime(2026, 6, 24, 8, 24, 5, 710296, tzinfo=datetime.timezone.utc)),
 HubSource(source_key='3cc8bb38-0301-4ae5-a489-c01891e8a65a', source_id='gba', source_name='Gemeinsamer Bundesausschuss', created_at=datetime.datetime(2026, 6, 24, 8, 24, 5, 712593, tzinfo=datetime.timezone.utc)),
 HubSource(source_key='3f17e13d-663c-4c07-a0fe-2c478bb04122', source_id='bundestag', source_name='Bundestag Open Data', created_at=datetime.datetime(2026, 6, 24, 8, 24, 5, 713180, tzinfo=datetime.timezone.utc)),
 HubSource(source_key='4759d414-fce6-4339-a5ae-2b49532108b2', source_id='bmg-web', source_name='Bundesministerium für Gesundheit', created_at=datetime.datetime(2026, 6, 24, 8, 24, 5, 714265, tzinfo=datetime.timezone.utc))]

In [4]:
import pandas as pd

hub_sources_df = pd.DataFrame([
    {
        "source_key": item.source_key,
        "source_id": item.source_id,
        "source_name": item.source_name,
        "created_at": item.created_at,
    }
    for item in hub_sources
])

hub_sources_df

,source_key,source_id,source_name,created_at
0,31dfd5cb-98b9-438a-92ea-bbad81849ed4,gesetze-im-internet,Gesetze im Internet,2026-06-24 08:24:05.710296+00:00
1,3cc8bb38-0301-4ae5-a489-c01891e8a65a,gba,Gemeinsamer Bundesausschuss,2026-06-24 08:24:05.712593+00:00
2,3f17e13d-663c-4c07-a0fe-2c478bb04122,bundestag,Bundestag Open Data,2026-06-24 08:24:05.713180+00:00
3,4759d414-fce6-4339-a5ae-2b49532108b2,bmg-web,Bundesministerium für Gesundheit,2026-06-24 08:24:05.714265+00:00


In [5]:
from app.repositories.source_registry import SourceRegistry
from app.repositories.document_repository import FileDocumentRepository
from app.services.incremental_ingestion_service import IncrementalIngestionService

registry = SourceRegistry(registry_path=str(PROJECT_ROOT / "data" / "source_registry.yaml"))
document_repo = FileDocumentRepository()

ingestion_service = IncrementalIngestionService(
    source_registry=registry,
    document_repository=document_repo,
)

summary = ingestion_service.ingest_source("gesetze-im-internet")
summary

IngestionSummary(source_id='gesetze-im-internet', fetched=3, ingested=0, skipped=3, updated=0, errors=0)

In [6]:
from app.repositories.document_repository import FileDocumentRepository
from app.repositories.hub_source_repository import FileHubSourceRepository
from app.repositories.hub_document_repository import FileHubDocumentRepository
from app.services.hub_document_service import HubDocumentService

document_repo = FileDocumentRepository()
source_repo = FileHubSourceRepository()
hub_doc_repo = FileHubDocumentRepository()

doc_service = HubDocumentService(
    document_repository=document_repo,
    hub_source_repository=source_repo,
    hub_document_repository=hub_doc_repo,
)

created_docs = doc_service.sync_documents()
print(created_docs)

[]


In [7]:
hub_documents = hub_doc_repo.load_all()
hub_documents

[HubDocument(document_key='edb129e090b0f32bd1bb003b48914ff469851caf19a4324b9a612fcdb711b3c4', document_id='0af5293a-fdcf-4b00-9584-2a8f6e36f786', source_key='31dfd5cb-98b9-438a-92ea-bbad81849ed4', source_id='gesetze-im-internet', document_name='file:///Users/rodrigue.lawson/VSCode%20Projects/lexcare-ai/data/raw/gesetze_im_internet/SGB_5.pdf', source_path='file:///Users/rodrigue.lawson/VSCode%20Projects/lexcare-ai/data/raw/gesetze_im_internet/SGB_5.pdf', created_at=datetime.datetime(2026, 6, 24, 20, 35, 10, 345944, tzinfo=datetime.timezone.utc)),
 HubDocument(document_key='429fef762c1caf90c4276ba677dbc4c79cfd564ef49ea2bd134b63399c37b167', document_id='a86b2a55-d52d-44b9-afb9-38ffc070f947', source_key='31dfd5cb-98b9-438a-92ea-bbad81849ed4', source_id='gesetze-im-internet', document_name='file:///Users/rodrigue.lawson/VSCode%20Projects/lexcare-ai/data/raw/gesetze_im_internet/SGB_11.pdf', source_path='file:///Users/rodrigue.lawson/VSCode%20Projects/lexcare-ai/data/raw/gesetze_im_internet/S